# Exploración y modelos (football dataset) — versión **todo en uno**

Para dictar por **módulos**, usa la secuencia **`01` → `06`** en `notebooks/README.md`.

Este archivo concentra clasificación, regresión y explicabilidad en un solo flujo (útil como demo rápida o repaso).

## Metodología (CRISP-DM)

- **CRISP-DM + métricas:** `docs/guias/crispdm_y_machine_learning.md`
- **Modelos y flujo integrado:** `docs/guias/modelos_y_flujo_integrado.md`

## Entorno

- Arranca Jupyter desde la **raíz del proyecto**: `kedro jupyter notebook` o `kedro jupyter lab`.
- Kernel **Kedro (analisis_equipos_de_football)** o **`.venv/bin/python`**.
- Necesitas `data/raw/database.sqlite` en local (no va en git).
- Reproducibilidad: **Kedro** `kedro run` (ver `README.md`).


In [1]:
%load_ext kedro.ipython

The kedro.ipython extension is already loaded. To reload it, use:
  %reload_ext kedro.ipython


In [2]:
catalog


'country': kedro_datasets.pandas.sql_dataset.SQLTableDataset
'league': kedro_datasets.pandas.sql_dataset.SQLTableDataset
'match': kedro_datasets.pandas.sql_dataset.SQLTableDataset
'player': kedro_datasets.pandas.sql_dataset.SQLTableDataset
'player_attributes': kedro_datasets.pandas.sql_dataset.SQLTableDataset
'team': kedro_datasets.pandas.sql_dataset.SQLTableDataset
'team_attributes': kedro_datasets.pandas.sql_dataset.SQLTableDataset
'parameters': kedro.io.memory_dataset.MemoryDataset(data='<dict>')

## Machine learning: resultado del partido

Clases: **victoria visitante (0)** / **empate (1)** / **victoria local (2)**.  
Features iniciales: cuotas **Bet365** (`B365H`, `B365D`, `B365A`). Luego comparamos varios algoritmos con las mismas particiones `train/test`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

match = catalog.load("match")

goal_diff = match["home_team_goal"] - match["away_team_goal"]
outcome = np.select(
    [goal_diff > 0, goal_diff == 0, goal_diff < 0],
    [2, 1, 0],
)

feat_cols = ["B365H", "B365D", "B365A"]
df = match[feat_cols].assign(outcome=outcome).dropna()

X = df[feat_cols]
y = df["outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Muestras: train={len(X_train):,}  test={len(X_test):,}")
print("Distribución de clases (train):", y_train.value_counts().sort_index().to_dict())

In [ ]:
models = {
    "LogisticRegression": Pipeline(
        [
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(max_iter=2_000, solver="lbfgs")),
        ]
    ),
    "LinearSVC": Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "clf",
                LinearSVC(dual=False, max_iter=5_000, random_state=42),
            ),
        ]
    ),
    "KNN_k15": Pipeline(
        [
            ("scale", StandardScaler()),
            ("clf", KNeighborsClassifier(n_neighbors=15, weights="distance")),
        ]
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        max_depth=16,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=200,
        max_depth=6,
        learning_rate=0.08,
        random_state=42,
    ),
}

rows = []
fitted = {}
predictions = {}

for name, est in models.items():
    est.fit(X_train, y_train)
    pred = est.predict(X_test)
    fitted[name] = est
    predictions[name] = pred
    rows.append(
        {
            "modelo": name,
            "accuracy": accuracy_score(y_test, pred),
            "f1_macro": f1_score(y_test, pred, average="macro"),
            "f1_weighted": f1_score(y_test, pred, average="weighted"),
        }
    )

resultados = pd.DataFrame(rows).sort_values("f1_macro", ascending=False).reset_index(
    drop=True
)
resultados

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(resultados))
w = 0.25
ax.bar(x - w, resultados["accuracy"], width=w, label="accuracy")
ax.bar(x, resultados["f1_macro"], width=w, label="f1_macro")
ax.bar(x + w, resultados["f1_weighted"], width=w, label="f1_weighted")
ax.set_xticks(x)
ax.set_xticklabels(resultados["modelo"], rotation=20, ha="right")
ax.set_ylim(0, 1)
ax.legend()
ax.set_title("Comparación de modelos (mismas features y split)")
plt.tight_layout()
plt.show()

In [ ]:
mejor = resultados.iloc[0]["modelo"]
y_pred_mejor = predictions[mejor]
print(f"Mejor según f1_macro: {mejor}\n")
print(
    classification_report(
        y_test,
        y_pred_mejor,
        target_names=["away_win", "draw", "home_win"],
    )
)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_mejor,
    display_labels=["away", "draw", "home"],
    ax=ax,
    colorbar=False,
)
ax.set_title(f"Matriz de confusión — {mejor}")
plt.tight_layout()
plt.show()

## Regresión: predecir goles del local

Objetivo continuo: `home_team_goal`. Las cuotas **Bet365** condensan expectativas del mercado; contrastamos modelos lineales (Ridge) frente a ensambles de árboles. **MAE** = error típico en goles; **RMSE** penaliza más los fallos grandes; **R²** indica cuánta varianza de los goles explica el modelo (útil como referencia, no como única verdad).

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import StandardScaler

y_reg = df["home_team_goal"]
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

reg_models = {
    "Ridge": SkPipeline(
        [("scale", StandardScaler()), ("reg", Ridge(alpha=1.0))]
    ),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=200,
        max_depth=16,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    ),
    "HistGradientBoostingRegressor": HistGradientBoostingRegressor(
        max_iter=200,
        max_depth=6,
        learning_rate=0.08,
        random_state=42,
    ),
}

reg_rows = []
fitted_reg = {}
for name, est in reg_models.items():
    est.fit(X_train_r, y_train_r)
    pred_r = est.predict(X_test_r)
    fitted_reg[name] = est
    reg_rows.append(
        {
            "modelo": name,
            "mae": mean_absolute_error(y_test_r, pred_r),
            "rmse": mean_squared_error(y_test_r, pred_r) ** 0.5,
            "r2": r2_score(y_test_r, pred_r),
        }
    )

resultados_reg = pd.DataFrame(reg_rows).sort_values("r2", ascending=False).reset_index(
    drop=True
)
resultados_reg

### Explicar modelos (clasificación y regresión)

- **Importancia por permutación**: se barajan valores de una columna en el conjunto de prueba y se mide cuánto empeora la métrica. Si casi no cambia, esa variable no aporta en presencia de las demás.
- **Coeficientes (logística / Ridge)**: en modelos lineales escalados, el signo y magnitud indican dirección y fuerza; no son “causa”, solo asociación en estos datos.
- **SHAP** (opcional, `pip install -e ".[explain]"`): valores Shapley aproximados; muy útil con árboles (`shap.TreeExplainer`). Requiere la dependencia extra `shap`.

In [ ]:
from IPython.display import display

mejor_clf = fitted[resultados.iloc[0]["modelo"]]
perm_clf = permutation_importance(
    mejor_clf,
    X_test,
    y_test,
    n_repeats=15,
    random_state=42,
    n_jobs=-1,
)
imp_clf = (
    pd.DataFrame(
        {
            "feature": feat_cols,
            "importance_mean": perm_clf.importances_mean,
            "importance_std": perm_clf.importances_std,
        }
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)
print("Clasificación — importancia por permutación (mejor modelo):")
display(imp_clf)

mejor_reg_name = resultados_reg.iloc[0]["modelo"]
mejor_reg = fitted_reg[mejor_reg_name]
perm_reg = permutation_importance(
    mejor_reg,
    X_test_r,
    y_test_r,
    n_repeats=15,
    random_state=42,
    n_jobs=-1,
)
imp_reg = (
    pd.DataFrame(
        {
            "feature": feat_cols,
            "importance_mean": perm_reg.importances_mean,
            "importance_std": perm_reg.importances_std,
        }
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)
print(f"Regresión — importancia por permutación ({mejor_reg_name}):")
display(imp_reg)

In [ ]:
try:
    import shap
except ImportError:
    print('Instala SHAP con: pip install -e ".[explain]"')
else:
    tree_est = fitted.get("RandomForest") or fitted.get("HistGradientBoosting")
    if tree_est is not None:
        sample = X_test.sample(min(400, len(X_test)), random_state=42)
        explainer = shap.TreeExplainer(tree_est)
        shap_vals = explainer.shap_values(sample)
        # Multiclase: lista de matrices (una por clase); mostramos victoria local (índice 2).
        vals = shap_vals[2] if isinstance(shap_vals, list) else shap_vals
        shap.summary_plot(
            vals,
            sample,
            feature_names=feat_cols,
            show=True,
        )
    else:
        print("No hay RandomForest/HistGradientBoosting en `fitted` para TreeExplainer.")

### Próximos pasos

- Más columnas de `match` (otras casas de apuestas) o joins con `team` / `team_attributes`.
- Validación **por temporada** (`season`) para no mezclar futuro y pasado.
- Afinar hiperparámetros con `GridSearchCV` o `RandomizedSearchCV` sobre el modelo que mejor salga aquí.
